# Stap 4 - Closed loop
Upload eenmalig [Stap_4_Closed_loop.ino](arduino/Stap_4_Closed_loop/Stap_4_Closed_loop.ino). Voer daarna de cellen van boven naar beneden uit.


In [ ]:
import csv
import importlib
import pandas as pd
import closed_loop.meting as meting

com_poort = "COM4"

## 1. Hoek meten
Laat de motorvoeding uit. Noteer de ruwe sensorhoek in rust als `beginhoek`.


In [ ]:
hoekmetingen = meting.test_hoek(com_poort=com_poort, meetduur=1.0, delta_t=0.1,opslaan=False)

## 2. PID instellen
De werkhoek is `sensorhoek - beginhoek`. Controleer dat deze toeneemt bij optillen.
`max_pwm` begrenst de uitgang (0?255); `delta_t` is het regelinterval in seconden.
Met `mode="fixed"` stopt de proef na `meetduur`; `"continuous"` loopt tot je de cel onderbreekt.


In [ ]:
beginhoek = 0  # Vervang door de gemeten sensorhoek in de nulstand.
doelhoek = 30.0    # graden ten opzichte van de ingestelde nulstand

kp = 0
ki = 0
kd = 0

max_pwm = 255     # bovengrens; kies een lagere waarde indien nodig
meetduur = 10.0   # seconden; stoptijd in fixed mode
delta_t = 0.02    # seconden (0.02-1); frequentie = 1 / delta_t Hz


mode = "fixed"   # "fixed" of "continuous" (stop met cel onderbreken)
toon_metingen = True  # False: geen meetpunten printen
opslaan = False  # True: meetdata en instellingen als CSV bewaren

metingen, csv_pad = meting.voer_pid_meting_uit(
    com_poort=com_poort, beginhoek=beginhoek,
    doelhoek=doelhoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm, toon_metingen=toon_metingen, opslaan=opslaan,
    meetduur=meetduur, delta_t=delta_t, mode=mode
)


PID: max_pwm=255, limiter=uit telwaarden/regelstap
PID-meting gestart.
   0.020 s | hoek   -0.000 graden | doel   40.000 graden | PWM   2 (  0.8%)
   0.040 s | hoek   -0.000 graden | doel   40.000 graden | PWM   5 (  2.0%)
   0.060 s | hoek   -0.000 graden | doel   40.000 graden | PWM   7 (  2.7%)
   0.080 s | hoek   -0.000 graden | doel   40.000 graden | PWM  10 (  3.9%)
   0.100 s | hoek   -0.000 graden | doel   40.000 graden | PWM  12 (  4.7%)
   0.120 s | hoek   -0.000 graden | doel   40.000 graden | PWM  14 (  5.5%)
   0.140 s | hoek   -0.000 graden | doel   40.000 graden | PWM  17 (  6.7%)
   0.160 s | hoek   -0.088 graden | doel   40.000 graden | PWM  19 (  7.5%)
   0.180 s | hoek   -0.000 graden | doel   40.000 graden | PWM  22 (  8.6%)
   0.200 s | hoek   -0.000 graden | doel   40.000 graden | PWM  24 (  9.4%)
   0.220 s | hoek   -0.000 graden | doel   40.000 graden | PWM  26 ( 10.2%)
   0.240 s | hoek   -0.000 graden | doel   40.000 graden | PWM  29 ( 11.4%)
   0.260 s | hoek

## 3. Meetdata bekijken
Opslaan staat standaard uit. Geef `opslaan=True` mee aan de PID-meting om een CSV in `data/` te bewaren. Zonder CSV toont de cel hieronder de meetpunten uit het geheugen.


In [10]:
bestand = csv_pad  # Of bijvoorbeeld "data/pid_20261027_....csv"

if bestand is None:
    df = pd.DataFrame(metingen, columns=["tijd", "doelhoek", "hoek", "pwm"])
    display(df.head())
else:

    # Lees de instellingen uit de commentaarheader.
    with open(bestand, encoding="utf-8", newline="") as f:
        csv_instellingen = {}
        for rij in csv.reader(f):
            if not rij or not rij[0].startswith("# "):
                break
            csv_instellingen[rij[0][2:]] = rij[1]

    # Pandas slaat de commentaarheader over en leest de meetkolommen.
    df = pd.read_csv(bestand, comment="#")
    display(pd.Series(csv_instellingen, name="Instelling").to_frame())
    display(df.head())

,tijd,doelhoek,hoek,pwm
0,0.02,40.0,-0.0,2.0
1,0.04,40.0,-0.0,5.0
2,0.06,40.0,-0.0,7.0
3,0.08,40.0,-0.0,10.0
4,0.10,40.0,-0.0,12.0


## 4. Hoekgrens instellen
`max_hoek` is standaard 180 graden. We kunnen echter deze hoekgrens ook naar beneden bijschroeven voor de veiligheid. Bij overschrijding stoppen motor en meting direct. Houd `doelhoek` binnen deze grens.


In [ ]:
beginhoek = 118.4  # Vervang door de gemeten sensorhoek in de nulstand.
doelhoek = 0    # graden ten opzichte van de ingestelde nulstand
max_hoek = 120.0  # Stopgrens voor de werkhoek; kies passend bij de opstelling.

kp = 0
ki = 0
kd = 0

max_pwm = 50     # bovengrens; kies een lagere waarde indien nodig
meetduur = 10.0   # seconden; stoptijd in fixed mode
delta_t = 0.02    # seconden (0.02-1); frequentie = 1 / delta_t Hz

mode = "fixed"   # "fixed" of "continuous" (stop met cel onderbreken)
toon_metingen = True  # False: geen meetpunten printen
opslaan = False  # True: meetdata en instellingen als CSV bewaren



metingen_veiligheid, csv_pad_veiligheid = meting.voer_pid_meting_uit(
    com_poort=com_poort, beginhoek=beginhoek,
    doelhoek=doelhoek, max_hoek=max_hoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm, toon_metingen=True,
    meetduur=meetduur, delta_t=delta_t, mode=mode,
)


PID: max_pwm=255, limiter=uit telwaarden/regelstap
PID-meting gestart.
   0.020 s | hoek  -11.261 graden | doel   15.000 graden | PWM  26 ( 10.2%)
   0.040 s | hoek  -10.998 graden | doel   15.000 graden | PWM  26 ( 10.2%)
   0.060 s | hoek  -10.558 graden | doel   15.000 graden | PWM  26 ( 10.2%)
   0.080 s | hoek  -10.031 graden | doel   15.000 graden | PWM  25 (  9.8%)
   0.100 s | hoek   -9.416 graden | doel   15.000 graden | PWM  24 (  9.4%)
   0.120 s | hoek   -8.713 graden | doel   15.000 graden | PWM  24 (  9.4%)
   0.140 s | hoek   -7.921 graden | doel   15.000 graden | PWM  23 (  9.0%)
   0.160 s | hoek   -7.043 graden | doel   15.000 graden | PWM  22 (  8.6%)
   0.180 s | hoek   -6.164 graden | doel   15.000 graden | PWM  21 (  8.2%)
   0.200 s | hoek   -5.109 graden | doel   15.000 graden | PWM  20 (  7.8%)
   0.220 s | hoek   -4.142 graden | doel   15.000 graden | PWM  19 (  7.5%)
   0.240 s | hoek   -3.088 graden | doel   15.000 graden | PWM  18 (  7.1%)
   0.260 s | hoek

RuntimeError: Arduino heeft de meting gestopt: ERROR:HOEKBEREIK

## 5. PWM-stap begrenzen
Vergelijk een proef zonder en met `max_pwm_stap`. Deze optie begrenst alleen de PWM-stijging per regelstap; dalen en stoppen gaan direct.
Zonder deze optie (of met `0`) staat de begrenzer uit. Gebruik voor beide proeven dezelfde instellingen.


In [ ]:
beginhoek = 115.4  # Vervang door de gemeten sensorhoek in de nulstand.
doelhoek = 90    # graden ten opzichte van de ingestelde nulstand
max_hoek = 180

kp = 3
ki = 3
kd = 3

max_pwm = 255     # bovengrens; kies een lagere waarde indien nodig
meetduur = 10.0   # seconden; stoptijd in fixed mode
delta_t = 0.02    # seconden (0.02-1); frequentie = 1 / delta_t Hz

max_pwm_stap =1

mode = "fixed"   # "fixed" of "continuous" (stop met cel onderbreken)
toon_metingen = True  # False: geen meetpunten printen
opslaan = False  # True: meetdata en instellingen als CSV bewaren

metingen_begrensd, csv_pad_begrensd = meting.voer_pid_meting_uit(
    com_poort=com_poort, beginhoek=beginhoek,
    doelhoek=doelhoek, max_hoek=max_hoek,
    kp=kp, ki=ki, kd=kd,
    max_pwm=max_pwm, toon_metingen=toon_metingen,opslaan=opslaan,
    meetduur=meetduur, delta_t=delta_t, mode=mode,max_pwm_stap=max_pwm_stap
)